# CALLIC quickstart — setup, data, train, eval
Faithful reproduction of arXiv:2412.17464v1. Works in Colab (GPU) or locally (CPU, slower).
Full recipe + 80h budget: `docs/plans/2026-09-12-lightning-80h-plan.md`.
> *Unofficial reproduction — code largely AI-generated; verify before trusting.*


In [ ]:
import os, sys
REPO_URL = 'https://github.com/hassenhamdi/callic.git'
def _find_repo():
    for c in [os.getcwd(), '/content']:
        root = c if os.path.basename(c) != 'callic' else os.path.dirname(c)
        if os.path.isdir(os.path.join(root, 'callic')):
            return root
    return None
ROOT = _find_repo()
if ROOT is None:
    import subprocess
    subprocess.run(['git', 'clone', REPO_URL, '/content/callic'], check=True)
    ROOT = '/content/callic'
    assert os.path.isdir(os.path.join(ROOT, 'callic')), f'clone lacks callic/ (got {os.listdir(ROOT)[:10]})'
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('repo root:', ROOT)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# 1. Dependencies (Colab-safe; no-op if already installed)
import importlib, subprocess
for pkg, mod in [('pillow', 'PIL'), ('numpy', 'numpy')]:
    try:
        importlib.import_module(mod); print(pkg, 'ok')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', pkg])
        print(pkg, 'installed')


In [ ]:
# 2. Quick data: Kodak 24 (eval) + DIV2K-valid 100 (demo training)
# Full paper data (DIV2K train + Flickr2K): bash tools/lightning_setup.sh
import urllib.request, subprocess
from pathlib import Path
kd = Path('data/eval/kodak'); kd.mkdir(parents=True, exist_ok=True)
GH = 'https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/master/PhotoCD_PCD0992'
for i in range(1, 25):
    n = f'{i:02d}'; dst = kd / f'kodim{n}.png'
    if not (dst.exists() and dst.stat().st_size > 10000):
        urllib.request.urlretrieve(f'{GH}/{n}.png', dst)
print('kodak:', len(list(kd.glob('*.png'))), '/24')
dv = Path('data/DIV2K_valid_HR')
if len(list(dv.glob('*.png'))) < 100:
    dv.mkdir(parents=True, exist_ok=True)
    subprocess.run('cd data && curl -L -o DIV2K_valid_HR.zip https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip && unzip -q -o DIV2K_valid_HR.zip && rm DIV2K_valid_HR.zip', shell=True)
print('div2k-valid:', len(list(dv.glob('*.png'))), '/100')


In [ ]:
# 3. Correctness gates: masks, CCI parity, merge equality, param budgets
import subprocess
r = subprocess.run([sys.executable, 'tests/test_faithful.py'], capture_output=True, text=True)
print(r.stdout[-300:] or r.stderr[-500:])
r = subprocess.run([sys.executable, 'tests/test_masks.py'], capture_output=True, text=True)
print(r.stdout[-300:] or r.stderr[-500:])


In [ ]:
# 3b. Latest weights from GitHub release — resume instead of training from scratch
# Release: https://github.com/hassenhamdi/callic/releases/tag/v0.1
# (74k-step MGCF + best@60500; keep weights out of git, fetch on demand)
# NOTE: torch.save files ARE zips under the hood — so try torch.load FIRST,
# and only unzip when the file is a plain archive of checkpoints.
import glob, subprocess, urllib.request, zipfile
REL_BASE = 'https://github.com/hassenhamdi/callic/releases/download/v0.1'
os.makedirs('checkpoints', exist_ok=True)
def _try_load(path):
    try:
        q = torch.load(path, map_location='cpu')
        sd = q['model'] if isinstance(q, dict) and 'model' in q else q
        return q if isinstance(q, dict) and 'model' in q else {'model': sd, 'step': '?', 'best_loss': None, 'best_step': None}
    except Exception:
        return None
have = sorted(glob.glob('checkpoints/*.pt'))
if not have:
    dl = 'checkpoints/_dl.bin'
    urllib.request.urlretrieve(f'{REL_BASE}/mgcf_74k.pt', dl)
    q = _try_load(dl)
    if q is not None:
        os.rename(dl, 'checkpoints/mgcf_drive.pt')
        print('release asset is a raw checkpoint -> checkpoints/mgcf_drive.pt')
    else:
        with open(dl, 'rb') as f:
            is_zip = f.read(2) == b'PK'
        assert is_zip, 'downloaded neither a checkpoint nor a zip — check the release'
        with zipfile.ZipFile(dl) as z:
            z.extractall('drive_pull')
        os.remove(dl)
        print('unzipped archive:')
        print(subprocess.run(['find', 'drive_pull', '-name', '*.pt'], capture_output=True, text=True).stdout)
RESUME_CANDIDATES = sorted(glob.glob('checkpoints/*.pt') + glob.glob('drive_pull/**/*.pt', recursive=True))
print('available ckpts:', RESUME_CANDIDATES)
RESUME_CKPT = 'checkpoints/mgcf_74k.pt' if os.path.exists('checkpoints/mgcf_74k.pt') else (RESUME_CANDIDATES[-1] if RESUME_CANDIDATES else None)
if RESUME_CKPT:
    p = torch.load(RESUME_CKPT, map_location='cpu')
    sd = p['model'] if isinstance(p, dict) and 'model' in p else p
    print(f'resume ckpt: {RESUME_CKPT} | step={p.get("step") if isinstance(p, dict) else "?"} '
          f'| best={p.get("best_loss") if isinstance(p, dict) else "?"}@{p.get("best_step") if isinstance(p, dict) else "?"} '
          f'| tensors={len(sd)}')
    print('Continue training with:')
    print(f"  nohup python tools/train.py --data data --steps 100000 --bs 32 --lr 5e-4 --schedule cosine --log-every 500 --keep-every 10000 --keep-last 3 --resume --out {RESUME_CKPT} > train.log 2>&1 &")
else:
    print('No checkpoint found — training will start from scratch.')


In [ ]:
# 4. Demo training: 200 steps on DIV2K-valid patches (~10 min T4, longer on CPU)
# Paper recipe is 2M steps on DIV2K+Flickr2K — see cell 6 and tools/train.py.
import subprocess
r = subprocess.run([sys.executable, 'tools/train.py', '--data', 'data/DIV2K_valid_HR',
                    '--steps', '200', '--bs', '32', '--schedule', 'cosine',
                    '--log-every', '50', '--keep-every', '0',
                    '--out', 'checkpoints/demo.pt'], capture_output=True, text=True)
print(r.stdout[-800:] or r.stderr[-800:])


In [ ]:
# 5. Eval demo ckpt on Kodak 24 (honest NLL bpsp, weight bits N/A for base)
import glob
import numpy as np
from PIL import Image
from callic.mgcf import MGCF
from callic.mixture import discretized_mixture_nll
m = MGCF()
m.load_state_dict(torch.load('checkpoints/demo.pt', map_location='cpu')['model'])
m.eval()
tot, n = 0.0, 0
with torch.no_grad():
    for f in sorted(glob.glob('data/eval/kodak/*.png')):
        a = torch.from_numpy(np.array(Image.open(f).convert('RGB'), dtype=np.uint8)).permute(2, 0, 1).unsqueeze(0)
        tot += float(discretized_mixture_nll(a, m(a.float())).item()); n += 1
print(f'demo Kodak mean bpsp={tot / n:.4f} (random-init ~23, trained-2000 ~5.3, paper MGCF 2.77)')


## 6. Full training (GPU, background)
Paper recipe — run in a terminal (not this kernel), resumable, best+numbered ckpts:
```bash
nohup python tools/train.py --data data --steps 2000000 --bs 32 --lr 5e-4 \
  --schedule cosine --log-every 500 --keep-every 50000 --keep-last 3 --resume \
  --out checkpoints/mgcf_full.pt > train.log 2>&1 &
```
Then per-image RPFT (CALLIC rows): `python tools/eval.py --ckpt checkpoints/mgcf_full.pt --data_root data/eval --rpft`.